---
## Part 2 — Ontology Discovery

Goal: inspect explanations empirically, then propose a first-pass set of scoreable response features for a future linear 4-way choice model.

In [1]:
# ── Cell 10: Load filtered parquet ────────────────────────────────────────

import pandas as pd
import numpy as np
import os, re, string
from collections import Counter

PARQUET_PATH = 'empirics_communityalignment/ca_filtered.parquet'
assert os.path.exists(PARQUET_PATH), f'File not found: {PARQUET_PATH}'

filt = pd.read_parquet(PARQUET_PATH)
print(f'Loaded: {len(filt):,} rows × {filt.shape[1]} columns')
print(f'Columns:\n{filt.columns.tolist()}')

Loaded: 9,407 rows × 46 columns
Columns:
['hf_split', 'conversation_id', 'annotator_id', 'wave', 'assigned_lang', 'is_pregenerated_first_prompt', 'annotator_age', 'annotator_gender', 'annotator_education_level', 'annotator_political', 'annotator_ethnicity', 'annotator_country', 'first_turn_prompt', 'first_turn_responses', 'first_turn_response_a', 'first_turn_response_b', 'first_turn_response_c', 'first_turn_response_d', 'first_turn_preferred_response', 'first_turn_feedback', 'second_turn_prompt', 'second_turn_responses', 'second_turn_response_a', 'second_turn_response_b', 'second_turn_response_c', 'second_turn_response_d', 'second_turn_preferred_response', 'second_turn_feedback', 'third_turn_prompt', 'third_turn_responses', 'third_turn_response_a', 'third_turn_response_b', 'third_turn_response_c', 'third_turn_response_d', 'third_turn_preferred_response', 'third_turn_feedback', 'fourth_turn_prompt', 'fourth_turn_responses', 'fourth_turn_response_a', 'fourth_turn_response_b', 'fourth_tur

In [2]:
# ── Cell 11: Schema inspection — locate relevant columns ──────────────────
#
# We need four groups of columns:
#   P)  the prompt text shown to the model
#   R)  the four response texts (first_turn_response_a/b/c/d)
#   C)  which response was chosen (first_turn_preferred_response)
#   E)  the annotator's explanation (first_turn_explanation or similar)
#
# We discover these empirically rather than hardcoding them.

print('=== dtypes + null rates ===')
info = pd.DataFrame({
    'dtype':    filt.dtypes.astype(str),
    'null_pct': (filt.isnull().mean() * 100).round(1),
    'nunique':  filt.nunique(),
    'example':  filt.apply(lambda s: str(s.dropna().iloc[0])[:80] if s.notna().any() else 'ALL NULL'),
})
print(info.to_string())

# ── Heuristic column finder ───────────────────────────────────────────────
def find_cols(df, patterns):
    """Return columns whose names match any of the given regex patterns."""
    return [c for c in df.columns if any(re.search(p, c, re.I) for p in patterns)]

prompt_cols   = find_cols(filt, [r'prompt', r'question', r'context', r'input'])
response_cols = find_cols(filt, [r'response_[abcd]'])
preferred_col = find_cols(filt, [r'preferred'])
explain_cols  = find_cols(filt, [r'explanation', r'reason', r'rationale', r'comment'])

print('\n=== Candidate columns by role ===')
print(f'  prompt:      {prompt_cols}')
print(f'  responses:   {response_cols}')
print(f'  preferred:   {preferred_col}')
print(f'  explanation: {explain_cols}')

=== dtypes + null rates ===
                                 dtype  null_pct  nunique                                                                             example
hf_split                        object       0.0        1                                                                               train
conversation_id                  int64       0.0     9407                                                                     702992025732060
annotator_id                     int64       0.0      598                                                                      61575131153481
wave                             int64       0.0        2                                                                                   1
assigned_lang                   object       0.0        1                                                                                  en
is_pregenerated_first_prompt      bool       0.0        1                                                               

In [4]:
# ── Cell 12: Pin column names — EDIT if Cell 11 shows different names ──────
#
# Assign each role to a single column. If find_cols returned multiple
# candidates, pick the most specific one here.

COL_PROMPT    = 'first_turn_prompt'          # the text of the prompt/question
COL_RESP_A    = 'first_turn_response_a'
COL_RESP_B    = 'first_turn_response_b'
COL_RESP_C    = 'first_turn_response_c'
COL_RESP_D    = 'first_turn_response_d'
COL_PREFERRED = 'first_turn_preferred_response'   # 'response_a'…'response_d'
COL_EXPLAIN   = 'first_turn_feedback'

RESP_COLS = {
    'response_a': COL_RESP_A,
    'response_b': COL_RESP_B,
    'response_c': COL_RESP_C,
    'response_d': COL_RESP_D,
}

# ── Sanity check ──────────────────────────────────────────────────────────
needed = [COL_PROMPT, COL_RESP_A, COL_RESP_B, COL_RESP_C, COL_RESP_D,
          COL_PREFERRED, COL_EXPLAIN]
missing = [c for c in needed if c not in filt.columns]
if missing:
    raise ValueError(f'Columns not in parquet: {missing}\nAvailable: {filt.columns.tolist()}')

print('All columns found.')

# ── Add a convenience column: text of the chosen response ─────────────────
def get_chosen_text(row):
    col = RESP_COLS.get(row[COL_PREFERRED])
    return row[col] if col else None

filt = filt.copy()
filt['chosen_text'] = filt.apply(get_chosen_text, axis=1)

print(f'Rows with explanation:  {filt[COL_EXPLAIN].notna().sum():,}')
print(f'Rows with chosen text:  {filt["chosen_text"].notna().sum():,}')
print(f'Preferred distribution:\n{filt[COL_PREFERRED].value_counts().to_string()}')

All columns found.
Rows with explanation:  9,407
Rows with chosen text:  9,407
Preferred distribution:
response_d    3377
response_a    3010
response_c    1562
response_b    1458


In [5]:
# ── Cell 13: Sample explanation-bearing rows ──────────────────────────────
#
# Pull rows that have both a non-empty explanation and all four response texts.
# These are the rows we'll read manually and mine for n-grams.

has_all = (
    filt[COL_EXPLAIN].notna() &
    filt[COL_RESP_A].notna() &
    filt[COL_RESP_B].notna() &
    filt[COL_RESP_C].notna() &
    filt[COL_RESP_D].notna()
)
expl_df = filt[has_all].reset_index(drop=True)
print(f'Rows with explanation + all 4 responses: {len(expl_df):,}')

print('\n=== Explanation length distribution (chars) ===')
expl_df['expl_len'] = expl_df[COL_EXPLAIN].str.len()
print(expl_df['expl_len'].describe().round(1).to_string())

print('\n=== 10 sampled explanations ===')
sample_idx = expl_df.sample(10, random_state=42).index
for i in sample_idx:
    pref = expl_df.loc[i, COL_PREFERRED]
    expl = expl_df.loc[i, COL_EXPLAIN]
    print(f'\n[{i}] chose={pref}')
    print(f'  {expl[:300]}')

Rows with explanation + all 4 responses: 9,407

=== Explanation length distribution (chars) ===
count    9407.0
mean      341.0
std       211.7
min        20.0
25%       207.0
50%       301.0
75%       426.0
max      3497.0

=== 10 sampled explanations ===

[4252] chose=response_a
  Response A is clear, engaging, and well-structured, directly addressing the prompt by recommending specific indie music venues (The Echo, Echoplex, Bootleg Theater, Moroccan Lounge) with descriptive details like âintimate performances in a converted warehouse spaceâ for Bootleg and âornate dec

[5168] chose=response_a
  All the responses provide excellent answers with very helpful suggestions. My least favorite is D- while it does list several helpful suggestions that are found in the other responses, it does not expand on any of them and explain why they are helpful. A is the best as it highlights the most importa

[1123] chose=response_d
  Out of all the responses, response D gives a properly format

In [6]:
# ── Cell 14: Manual review table ──────────────────────────────────────────
#
# Build a flat table: one row per annotation, with the prompt, all four
# response texts, the chosen label, the chosen text, and the explanation.
# Truncate response text to keep the table readable.
#
# Use this table to read a sample and sanity-check the schema before mining.

N_REVIEW = 20          # number of examples to display
TRUNC    = 300         # characters per response before truncating

review_cols = [COL_PROMPT, COL_RESP_A, COL_RESP_B, COL_RESP_C, COL_RESP_D,
               COL_PREFERRED, 'chosen_text', COL_EXPLAIN]
review = (
    expl_df[review_cols]
    .sample(N_REVIEW, random_state=7)
    .reset_index(drop=True)
)

# truncate long text fields for display
for c in [COL_PROMPT, COL_RESP_A, COL_RESP_B, COL_RESP_C, COL_RESP_D,
          'chosen_text', COL_EXPLAIN]:
    review[c] = review[c].astype(str).str[:TRUNC]

pd.set_option('display.max_colwidth', TRUNC)
review

,first_turn_prompt,first_turn_response_a,first_turn_response_b,first_turn_response_c,first_turn_response_d,first_turn_preferred_response,chosen_text,first_turn_feedback
0,What are the best restaurants in Strasbourg for trying Alsatian cuisine?,"For an authentic Alsatian dining experience in Strasbourg, head to Le Tire Bouchon, a cozy restaurant serving traditional dishes like tarte flambée and baeckeoffe. Another highly-recommended spot is La Maison des Tanneurs, which offers a range of Alsatian specialties, including foie gras and pre...","Alsatian cuisine is all about hearty, comforting dishes that warm the soul, and Strasbourg has plenty of restaurants that deliver. One of the best places to try classic Alsatian fare is Au Pont Corbeau, a rustic restaurant located in the heart of the city's historic center. Their menu features a...","Strasbourg's culinary scene is a vibrant reflection of its cultural heritage, and Alsatian cuisine is a big part of that. To experience the best of Alsatian food, visit Le Maitre Renard, a Michelin-starred restaurant that offers a contemporary take on traditional Alsatian dishes. Their menu feat...","For Alsatian cuisine in Strasbourg, try:\n\n1. Maison Kammerzell: Traditional dishes in a historic setting.\n2. Le Tire Bouchon: Cozy restaurant with regional specialties.\n3. La Meilleure Vie: Modern take on Alsatian cuisine.\n4. Le Grand Colbert: Upscale dining with Alsatian flavors.\n5. Chez ...",response_a,"For an authentic Alsatian dining experience in Strasbourg, head to Le Tire Bouchon, a cozy restaurant serving traditional dishes like tarte flambée and baeckeoffe. Another highly-recommended spot is La Maison des Tanneurs, which offers a range of Alsatian specialties, including foie gras and pre...","I chose Response A because it highlighted a good mix of traditional and modern Alsatian restaurants, while also mentioning specific dishes like tarte flambee and baeckeoffe. The descriptions gave a sense of atmosphere, which made it feel more immersive.\nResponse D was a close second, it provide..."
1,"I'm attending my niece's 5th birthday party and want to write a gift card message, referencing how I've been practicing singing her favorite kid's song and suggesting we have a sing-along together during the celebration.","Here's a message idea for the gift card:\n\n""Happy 5th birthday to my amazing niece! I've been practicing my singing skills, and I'm excited to say that I've finally mastered your favorite kid's song. I'd love to have a sing-along with you during the party today! Let's get together and belt out ...","Here's a message idea for the gift card:\n\n""Happy 5th birthday to my amazing niece! I've been practicing my singing skills, and I'm excited to say that I've finally mastered your favorite kid's song. I think it's time for a sing-along! Let's get together during the party and belt out the tunes....","Here's a message idea for the gift card:\n\n""Happy 5th birthday to my amazing niece! I've been practicing my singing skills, and I'm excited to say that I've finally mastered your favorite kid's song. I think it's time for a sing-along! Let's get together during the party and belt out the tune t...","Here's a message idea for the gift card:\n\n""Happy 5th birthday to my amazing niece! I've been practicing my singing skills, and I'm excited to say that I've finally mastered your favorite kid's song. I'd love to have a sing-along with you today at the party! Let's belt out the tunes and have so...",response_a,"Here's a message idea for the gift card:\n\n""Happy 5th birthday to my amazing niece! I've been practicing my singing skills, and I'm excited to say that I've finally mastered your favorite kid's song. I'd love to have a sing-along with you during the party today! Let's get together and belt out ...","This is the most preferred response because it feels the most heartfelt and playful. It balances enthusiasm(""Let's belt out the tunes and have some fun together"") with warm and lo

In [7]:
# ── Cell 15: N-gram / phrase analysis over explanations ───────────────────
#
# We tokenize all explanations and count unigrams, bigrams, and trigrams
# after removing punctuation and a conservative English stopword list.
# No embeddings, no LDA — just counting.
#
# Read the output carefully before writing the ontology: the most frequent
# phrases are the vocabulary annotators actually use, which should map
# directly to scoreable features.

STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do',
    'does','did','it','its','this','that','these','those','i','you','he',
    'she','we','they','me','him','her','us','them','my','your','his','our',
    'their','what','which','who','how','why','when','where','not','no',
    'more','most','also','than','so','as','if','would','could','should',
    'will','can','may','might','just','very','too','much','many','some',
    'all','any','other','one','two','three','four','response','answer',
    'option','choice','because','while','though','both','each','even',
    'from','about','up','out','there','here','then','than','into','over',
    'only','like','since','however','although','therefore','thus',
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[%s]" % re.escape(string.punctuation), ' ', text)
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 2]

def ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

explanations = expl_df[COL_EXPLAIN].dropna().astype(str).tolist()
print(f'Analysing {len(explanations):,} explanations...')

uni_ctr, bi_ctr, tri_ctr = Counter(), Counter(), Counter()
for expl in explanations:
    toks = tokenize(expl)
    uni_ctr.update(toks)
    bi_ctr.update(ngrams(toks, 2))
    tri_ctr.update(ngrams(toks, 3))

TOP = 40
print(f'\n=== Top {TOP} unigrams ===')
for w, c in uni_ctr.most_common(TOP):
    print(f'  {c:6,}  {w}')

print(f'\n=== Top {TOP} bigrams ===')
for w, c in bi_ctr.most_common(TOP):
    print(f'  {c:6,}  {w}')

print(f'\n=== Top {TOP} trigrams ===')
for w, c in tri_ctr.most_common(TOP):
    print(f'  {c:6,}  {w}')

Analysing 9,407 explanations...

=== Top 40 unigrams ===
   2,102  responses
   1,992  preferred
   1,861  well
   1,771  provides
   1,765  best
   1,571  prompt
   1,424  gives
   1,364  good
   1,342  user
   1,305  clear
   1,262  information
   1,236  makes
   1,146  specific
   1,090  list
   1,087  easy
   1,033  making
   1,031  prefer
     973  practical
     831  helpful
     809  second
     796  options
     789  offers
     789  such
     759  way
     750  personal
     718  without
     709  feel
     667  details
     666  tone
     648  structured
     635  experience
     634  concise
     612  suggestions
     611  mentions
     604  feels
     590  different
     585  make
     578  provided
     558  includes
     552  detailed

=== Top 40 bigrams ===
     257  well written
     245  easy read
     240  well structured
     238  easy follow
     226  well rounded
     178  easy understand
     163  provides clear
     163  first preferred
     157  balance between


In [8]:
# ── Cell 15b: Keyword-in-context (KWIC) for candidate feature terms ────────
#
# For each plausible feature keyword, show 5 explanation snippets that
# contain it. This lets you verify whether the word is being used in a
# response-describing sense or in some other sense (e.g. "helpful" applied
# to the annotator, not the response).

KWIC_TERMS = [
    'specific', 'detail', 'clear', 'concise', 'complete', 'accurate',
    'relevant', 'helpful', 'tone', 'formal', 'friendly', 'direct',
    'example', 'structure', 'organized', 'safe', 'appropriate', 'fluent',
    'coherent', 'actionable', 'step', 'practical',
]
KWIC_N = 5          # snippets per term
WINDOW = 60         # chars either side of match

print('=== Keyword-in-context snippets ===\n')
for term in KWIC_TERMS:
    hits = []
    for expl in explanations:
        for m in re.finditer(r'\b' + re.escape(term) + r'\w*', expl, re.I):
            s, e = m.start(), m.end()
            snip = expl[max(0, s-WINDOW):e+WINDOW].replace('\n', ' ')
            hits.append(f'  …{snip}…')
            if len(hits) >= KWIC_N:
                break
        if len(hits) >= KWIC_N:
            break
    print(f'"{term}"  ({len(hits)} snippets shown)')
    for h in hits[:KWIC_N]:
        print(h)
    print()

=== Keyword-in-context snippets ===

"specific"  (5 snippets shown)
  …er.  Response D is my second preference as it also provides specific, actionable tips related to different hair types. While it'…
  …ence because it is the only response that actually lays out specific models of projectors to buy (Anker Nebula Capsule or Epson …
  …ost approachable welcome to the new hires of the company. I specifically like the last sentence closing the paragraph that almost pe…
  … wide range of options from various food groups and is very specific in its details. Response D is close, but Response A gives m…
  …od I am looking for. Response C is also good but I like the specific dishes Response B offered and I prefer a restaurant rather …

"detail"  (5 snippets shown)
  …s in their day-to-day life, like when it states "thoughtful details that you weave into our daily life." Response B is the most…
  …Response A has provided detailed information about the German language proficiency required …
  … 

In [9]:
# ── Cell 16: First-pass ontology of response features ─────────────────────
#
# BASIS: the n-gram counts and KWIC snippets above.
# REVISION POLICY: after running Cells 15/15b, delete features whose
# keywords never appear in the KWIC output, and add any high-frequency
# phrases that are missing.
#
# Design constraints:
#   - Each feature describes a property of ONE response given ONE prompt.
#   - Each feature is scoreable on a 0–1 or low/high scale by a human
#     (or eventually a linear model) without seeing the other responses.
#   - No annotator-level properties (age, politics, etc.).
#   - No latent/unobserved constructs.

ontology = [
    {
        'feature':      'instruction_adherence',
        'definition':   'The response directly addresses the specific task or question posed in the prompt, without ignoring subparts or substituting a different question.',
        'high':         'Answers exactly what was asked, covers all subparts.',
        'low':          'Answers a related but different question, or ignores key constraints in the prompt.',
        'confusable_with': 'completeness (adherence is about being on-task; completeness is about depth of coverage within task)',
        'frequency_note': 'Expected high — "addresses", "answers the question", "relevant to" appear frequently.',
    },
    {
        'feature':      'specificity',
        'definition':   'The response provides concrete details, numbers, named entities, or step-by-step instructions rather than generic statements.',
        'high':         'Names specific tools, quantities, steps, or examples.',
        'low':          'Uses vague language: "it depends", "you should consider", "many options exist".',
        'confusable_with': 'completeness (a response can be long and complete yet still vague; specificity is about concreteness not length)',
        'frequency_note': 'Expected high — "specific", "detail", "example", "step" are common annotator words.',
    },
    {
        'feature':      'completeness',
        'definition':   'The response covers all logically necessary parts of the answer given the prompt, leaving no important sub-question unaddressed.',
        'high':         'Addresses every facet the prompt raises; a follow-up question would be unnecessary.',
        'low':          'Leaves out a sub-question or stops before a natural conclusion.',
        'confusable_with': 'instruction_adherence (completeness is about coverage depth; adherence is about being on-target at all)',
        'frequency_note': 'Expected moderate — "complete", "covers", "missing" appear but less than specificity terms.',
    },
    {
        'feature':      'clarity',
        'definition':   'The response is written in language that is easy to parse: simple vocabulary appropriate to the audience, short sentences, no ambiguous pronoun references.',
        'high':         'A non-expert could follow it without re-reading.',
        'low':          'Dense jargon, long tangled sentences, or ambiguous referents.',
        'confusable_with': 'coherence (clarity is sentence-level readability; coherence is paragraph/discourse-level logical flow)',
        'frequency_note': 'Expected high — "clear", "easy to understand", "simple" are frequent annotator phrases.',
    },
    {
        'feature':      'coherence',
        'definition':   'The response has a logical internal structure: ideas are ordered sensibly, transitions are present, and no contradictions appear within the text.',
        'high':         'Reading order matches logical order; conclusion follows from body.',
        'low':          'Ideas jump around, contradict each other, or repeat without adding content.',
        'confusable_with': 'clarity (coherence is about logical flow between ideas; clarity is about word/sentence legibility)',
        'frequency_note': 'Expected moderate — "organized", "structured", "flows" appear but less often than clarity terms.',
    },
    {
        'feature':      'conciseness',
        'definition':   'The response delivers needed information without unnecessary padding, repetition, or off-topic digressions.',
        'high':         'Same informational content as a longer response; no filler sentences.',
        'low':          'Repeats points already made, adds boilerplate caveats, pads with pleasantries.',
        'confusable_with': 'completeness (a concise response can still be complete; the trade-off only applies when content is actually omitted)',
        'frequency_note': 'Expected moderate — "concise", "brief", "to the point" appear; may overlap with clarity in annotations.',
    },
    {
        'feature':      'factual_plausibility',
        'definition':   'The factual claims in the response are consistent with broadly known facts and do not contain obvious errors or hallucinations.',
        'high':         'All stated facts are correct or at minimum not obviously wrong.',
        'low':          'Contains incorrect figures, impossible claims, or contradicts well-known facts.',
        'confusable_with': 'instruction_adherence (a response can be on-topic but factually wrong; these are orthogonal)',
        'frequency_note': 'Expected moderate — "accurate", "correct", "wrong" appear; may be harder to score without domain knowledge.',
    },
    {
        'feature':      'tone_fit',
        'definition':   'The formality, warmth, and register of the response match the implied context of the prompt (e.g. formal for professional queries, empathetic for personal struggles).',
        'high':         'Matches the social register of the prompt naturally.',
        'low':          'Inappropriately casual for a formal request, or clinically cold for an emotional one.',
        'confusable_with': 'safety/appropriateness (tone_fit is about register match, not about harm avoidance)',
        'frequency_note': 'Expected moderate — "tone", "formal", "friendly", "empathetic" appear; may be less common for factual prompts.',
    },
    {
        'feature':      'actionability',
        'definition':   'The response gives the user something concrete they can do next: a specific recommendation, step, command, or decision criterion.',
        'high':         'User can act immediately without further research.',
        'low':          'Presents tradeoffs or background without guiding toward a decision.',
        'confusable_with': 'specificity (actionability requires specificity, but a response can be specific about background facts without being actionable)',
        'frequency_note': 'Expected moderate — "practical", "actionable", "step", "recommend" appear; most useful for how-to prompts.',
    },
    {
        'feature':      'use_of_examples',
        'definition':   'The response illustrates its claims with at least one concrete example, analogy, or worked demonstration.',
        'high':         'Includes one or more examples that directly support the main point.',
        'low':          'Makes claims in the abstract with no illustration.',
        'confusable_with': 'specificity (examples are one way to achieve specificity, but specificity can also come from numbers or named entities without an example)',
        'frequency_note': 'Expected moderate to high — "example", "for instance", "such as" appear frequently in n-grams.',
    },
    {
        'feature':      'safety',
        'definition':   'The response avoids harmful, offensive, or inappropriate content and does not assist with clearly unethical requests.',
        'high':         'No harmful content; appropriate disclaimers where genuinely needed.',
        'low':          'Contains harmful instructions, discriminatory language, or reckless advice.',
        'confusable_with': 'tone_fit (safety is about harm avoidance; tone_fit is about register match — a safe response can still have the wrong tone)',
        'frequency_note': 'Expected low frequency in annotations (most responses are safe), but important to include as a near-zero baseline feature.',
    },
]

onto_df = pd.DataFrame(ontology)
print(f'{len(onto_df)} features proposed.\n')
print(onto_df[['feature','definition']].to_string(index=False))
onto_df

11 features proposed.

              feature                                                                                                                                                             definition
instruction_adherence                     The response directly addresses the specific task or question posed in the prompt, without ignoring subparts or substituting a different question.
          specificity                                          The response provides concrete details, numbers, named entities, or step-by-step instructions rather than generic statements.
         completeness                                       The response covers all logically necessary parts of the answer given the prompt, leaving no important sub-question unaddressed.
              clarity            The response is written in language that is easy to parse: simple vocabulary appropriate to the audience, short sentences, no ambiguous pronoun references.
            coherence           

,feature,definition,high,low,confusable_with,frequency_note
0,instruction_adherence,"The response directly addresses the specific task or question posed in the prompt, without ignoring subparts or substituting a different question.","Answers exactly what was asked, covers all subparts.","Answers a related but different question, or ignores key constraints in the prompt.",completeness (adherence is about being on-task; completeness is about depth of coverage within task),"Expected high — ""addresses"", ""answers the question"", ""relevant to"" appear frequently."
1,specificity,"The response provides concrete details, numbers, named entities, or step-by-step instructions rather than generic statements.","Names specific tools, quantities, steps, or examples.","Uses vague language: ""it depends"", ""you should consider"", ""many options exist"".",completeness (a response can be long and complete yet still vague; specificity is about concreteness not length),"Expected high — ""specific"", ""detail"", ""example"", ""step"" are common annotator words."
2,completeness,"The response covers all logically necessary parts of the answer given the prompt, leaving no important sub-question unaddressed.",Addresses every facet the prompt raises; a follow-up question would be unnecessary.,Leaves out a sub-question or stops before a natural conclusion.,instruction_adherence (completeness is about coverage depth; adherence is about being on-target at all),"Expected moderate — ""complete"", ""covers"", ""missing"" appear but less than specificity terms."
3,clarity,"The response is written in language that is easy to parse: simple vocabulary appropriate to the audience, short sentences, no ambiguous pronoun references.",A non-expert could follow it without re-reading.,"Dense jargon, long tangled sentences, or ambiguous referents.",coherence (clarity is sentence-level readability; coherence is paragraph/discourse-level logical flow),"Expected high — ""clear"", ""easy to understand"", ""simple"" are frequent annotator phrases."
4,coherence,"The response has a logical internal structure: ideas are ordered sensibly, transitions are present, and no contradictions appear within the text.",Reading order matches logical order; conclusion follows from body.,"Ideas jump around, contradict each other, or repeat without adding content.",clarity (coherence is about logical flow between ideas; clarity is about word/sentence legibility),"Expected moderate — ""organized"", ""structured"", ""flows"" appear but less often than clarity terms."
5,conciseness,"The response delivers needed information without unnecessary padding, repetition, or off-topic digressions.",Same informational content as a longer response; no filler sentences.,"Repeats points already made, adds boilerplate caveats, pads with pleasantries.",completeness (a concise response can still be complete; the trade-off only applies when content is actually omitted),"Expected moderate — ""concise"", ""brief"", ""to the point"" appear; may overlap with clarity in annotations."
6,factual_plausibility,The factual claims in the response are consistent with broadly known facts and do not contain obvious errors or hallucinations.,All stated facts are correct or at minimum not obviously wrong.,"Contains incorrect figures, impossible claims, or contradicts well-known facts.",instruction_adherence (a response can be on-topic but factually wrong; these are orthogonal),"Expected moderate — ""accurate"", ""correct"", ""wrong"" appear; may be harder to score without domain knowledge."
7,tone_fit,"The formality, warmth, and register of the response match the implied context of the prompt (e.g. formal for professional queries, empathetic for personal struggles).",Matches the social register of the prompt naturally.,"Inappropriately casual for a formal request, or clinically cold for an emotional one.","safety/appropriateness (tone_fit is about register match, not about harm avoidance)","Expected moderate — ""tone"", ""f

In [10]:
# ── Cell 17: Save ontology to disk ────────────────────────────────────────

import json

OUT_DIR = 'empirics_communityalignment'
os.makedirs(OUT_DIR, exist_ok=True)

CSV_PATH  = os.path.join(OUT_DIR, 'response_feature_ontology.csv')
JSON_PATH = os.path.join(OUT_DIR, 'response_feature_ontology.json')

onto_df.to_csv(CSV_PATH, index=False)
with open(JSON_PATH, 'w') as f:
    json.dump(ontology, f, indent=2)

print(f'Saved CSV  → {CSV_PATH}')
print(f'Saved JSON → {JSON_PATH}')
print(f'\n{len(onto_df)} features in ontology:')
for feat in onto_df['feature']:
    print(f'  • {feat}')

# ── Keyword coverage check ─────────────────────────────────────────────────
# For each feature, check whether its core keyword appears in the
# explanation corpus at all. A zero-count feature may need to be dropped
# or renamed to match how annotators actually talk.
print('\n=== Keyword corpus coverage ===')
FEATURE_KEYWORDS = {
    'instruction_adherence': [r'address', r'answer', r'relevant', r'question'],
    'specificity':           [r'specific', r'detail', r'concrete', r'example'],
    'completeness':          [r'complete', r'cover', r'missing', r'thorough'],
    'clarity':               [r'clear', r'simple', r'easy to understand', r'straightforward'],
    'coherence':             [r'coherent', r'organized', r'structured', r'flow'],
    'conciseness':           [r'concise', r'brief', r'point', r'wordy'],
    'factual_plausibility':  [r'accurate', r'correct', r'wrong', r'factual'],
    'tone_fit':              [r'tone', r'formal', r'friendly', r'empathetic', r'appropriate'],
    'actionability':         [r'actionable', r'practical', r'step', r'recommend'],
    'use_of_examples':       [r'example', r'instance', r'such as', r'illustrat'],
    'safety':                [r'safe', r'harm', r'appropriate', r'offensive'],
}
all_text = ' '.join(explanations).lower()
for feat, keywords in FEATURE_KEYWORDS.items():
    hits = {kw: len(re.findall(r'\b' + kw, all_text)) for kw in keywords}
    total = sum(hits.values())
    hit_str = '  '.join(f'{kw}={n:,}' for kw, n in hits.items())
    flag = '' if total > 50 else '  ← LOW COVERAGE — consider dropping'
    print(f'  {feat:25s}  total={total:5,}  {hit_str}{flag}')

Saved CSV  → empirics_communityalignment/response_feature_ontology.csv
Saved JSON → empirics_communityalignment/response_feature_ontology.json

11 features in ontology:
  • instruction_adherence
  • specificity
  • completeness
  • clarity
  • coherence
  • conciseness
  • factual_plausibility
  • tone_fit
  • actionability
  • use_of_examples
  • safety

=== Keyword corpus coverage ===
  instruction_adherence      total=2,958  address=446  answer=1,524  relevant=366  question=622
  specificity                total=3,767  specific=1,422  detail=1,641  concrete=131  example=573
  completeness               total=  876  complete=382  cover=383  missing=19  thorough=92
  clarity                    total=2,596  clear=1,859  simple=426  easy to understand=162  straightforward=149
  coherence                  total=1,201  coherent=77  organized=220  structured=648  flow=256
  conciseness                total=2,065  concise=690  brief=344  point=978  wordy=53
  factual_plausibility       tota

---
## Part 3 — Response-Level Representation & Scoring Rubric

### Why independent, absolute scoring matters

A linear 4-way choice model assigns utility $U(r,p) = \mathbf{x}_{r,p}^\top \boldsymbol{\beta}$
to response $r$ given prompt $p$, and predicts the annotator chooses the highest-utility response.

The feature vector $\mathbf{x}_{r,p}$ must describe **that single response in isolation**.
If any feature were defined relative to the other three options — e.g. "more specific than the
alternatives" — the feature value would change every time the choice set changed. That breaks
transferability: you could not score a new response without already knowing its three competitors.

**Scoring rule:** given only the prompt and one response, assign each feature a score.
The preferred label, the other responses, and the annotator's explanation are kept as audit columns
but must not influence feature scores.

In [11]:
# ── Cell 18: Schema confirmation for Part 3 ───────────────────────────────
#
# Restate the column roles we confirmed in Part 2 and flag one important
# structural fact: conversation_id is unique per row, so it is NOT the
# shared prompt unit. first_turn_prompt (1,008 unique texts) is.

COL_CONV_ID   = 'conversation_id'    # unique per row — identifies one annotator interaction
COL_ANNOTATOR = 'annotator_id'       # 598 unique annotators
COL_PROMPT    = 'first_turn_prompt'  # 1,008 unique texts — the SHARED prompt unit
COL_RESP_A    = 'first_turn_response_a'
COL_RESP_B    = 'first_turn_response_b'
COL_RESP_C    = 'first_turn_response_c'
COL_RESP_D    = 'first_turn_response_d'
COL_PREFERRED = 'first_turn_preferred_response'
COL_FEEDBACK  = 'first_turn_feedback'   # annotator free-text — AUDIT ONLY, not for scoring

RESP_POSITION_COLS = {
    'response_a': COL_RESP_A,
    'response_b': COL_RESP_B,
    'response_c': COL_RESP_C,
    'response_d': COL_RESP_D,
}
FEATURES = [
    'instruction_adherence', 'specificity', 'completeness', 'clarity',
    'coherence', 'conciseness', 'factual_plausibility', 'tone_fit',
    'actionability', 'use_of_examples', 'safety',
]

needed = [COL_CONV_ID, COL_ANNOTATOR, COL_PROMPT,
          COL_RESP_A, COL_RESP_B, COL_RESP_C, COL_RESP_D,
          COL_PREFERRED, COL_FEEDBACK]
missing = [c for c in needed if c not in filt.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print('Columns confirmed.')
print(f'\nKey cardinalities:')
for col in [COL_CONV_ID, COL_ANNOTATOR, COL_PROMPT,
            COL_RESP_A, COL_RESP_B, COL_RESP_C, COL_RESP_D]:
    print(f'  {col:35s}  nunique={filt[col].nunique():,}')
print(f'\nRows in filtered dataset: {len(filt):,}')
print(f'⚠  conversation_id is unique per row → NOT the shared prompt unit.')
print(f'   Use first_turn_prompt text (nunique={filt[COL_PROMPT].nunique():,}) as the grouping key.')

Columns confirmed.

Key cardinalities:
  conversation_id                      nunique=9,407
  annotator_id                         nunique=598
  first_turn_prompt                    nunique=1,008
  first_turn_response_a                nunique=2,817
  first_turn_response_b                nunique=2,822
  first_turn_response_c                nunique=2,821
  first_turn_response_d                nunique=2,818

Rows in filtered dataset: 9,407
⚠  conversation_id is unique per row → NOT the shared prompt unit.
   Use first_turn_prompt text (nunique=1,008) as the grouping key.


In [12]:
# ── Cell 19: Detect position shuffling ────────────────────────────────────
#
# In many preference studies, the A/B/C/D position labels are randomised per
# annotator to control for position bias. If that happened here, then
# "response_a" does NOT refer to the same response text across annotators for
# the same prompt — so we cannot deduplicate by (prompt, position).
#
# Diagnostic: for each prompt, count how many DISTINCT response texts appear
# in each position. If positions are fixed, each prompt should have exactly
# 1 distinct text per position. If shuffled, we'll see >1.

pos_consistency = {}
for pos, col in RESP_POSITION_COLS.items():
    unique_per_prompt = filt.groupby(COL_PROMPT)[col].nunique()
    pos_consistency[pos] = unique_per_prompt.describe()

consistency_df = pd.DataFrame(pos_consistency).T
print('Distinct response texts per position per prompt (should be 1 if fixed, >1 if shuffled):')
print(consistency_df[['min','mean','max']].round(2).to_string())

mean_texts = consistency_df['mean'].mean()
if mean_texts > 1.05:
    print(f'\n⚠  Mean distinct texts per position = {mean_texts:.2f} > 1.')
    print('   Positions ARE shuffled across annotators.')
    print('   Deduplication key must be (prompt_text, response_text), NOT (prompt, position).')
else:
    print(f'\n✓  Positions appear fixed (mean distinct texts ≈ {mean_texts:.2f}).')
    print('   Safe to deduplicate by (prompt, position).')

Distinct response texts per position per prompt (should be 1 if fixed, >1 if shuffled):
            min  mean   max
response_a  1.0  2.79  21.0
response_b  1.0  2.80  21.0
response_c  1.0  2.80  19.0
response_d  1.0  2.80  18.0

⚠  Mean distinct texts per position = 2.80 > 1.
   Positions ARE shuffled across annotators.
   Deduplication key must be (prompt_text, response_text), NOT (prompt, position).


In [18]:
# ── Cell 20: Filter degenerate rows, then melt to long format ─────────────
#
# A row is "degenerate" if all four response texts are identical — the
# annotator's choice is meaningless because there is nothing to choose between.
# Drop those rows before doing anything else.

import hashlib

def make_id(text, length=12):
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:length]

# ── All-4-same filter ─────────────────────────────────────────────────────
all_same = (
    (filt[COL_RESP_A] == filt[COL_RESP_B]) &
    (filt[COL_RESP_B] == filt[COL_RESP_C]) &
    (filt[COL_RESP_C] == filt[COL_RESP_D])
)
n_degenerate = all_same.sum()
print(f'Degenerate rows (all 4 responses identical): {n_degenerate:,}  →  dropping.')

filt_clean = filt[~all_same].reset_index(drop=True)
print(f'Rows remaining: {len(filt_clean):,}  (was {len(filt):,})')

# ── Melt to long format ───────────────────────────────────────────────────
# One row per annotator × response position.
# is_chosen flags the response the annotator actually picked.
# Audit columns (preferred, feedback) ride along for later validation only.

AUDIT_COLS = [COL_CONV_ID, COL_ANNOTATOR, COL_PREFERRED, COL_FEEDBACK]

long_frames = []
for pos_label, col in RESP_POSITION_COLS.items():
    chunk = filt_clean[[COL_PROMPT] + AUDIT_COLS + [col]].copy()
    chunk = chunk.rename(columns={col: 'response_text'})
    chunk['response_position'] = pos_label
    chunk['is_chosen'] = (chunk[COL_PREFERRED] == pos_label)
    long_frames.append(chunk)

long_df = pd.concat(long_frames, ignore_index=True)
long_df = long_df.dropna(subset=['response_text']).reset_index(drop=True)

print(f'\nLong-format table: {len(long_df):,} rows × {long_df.shape[1]} columns')
print(f'is_chosen distribution:\n{long_df["is_chosen"].value_counts().to_string()}')

Degenerate rows (all 4 responses identical): 0  →  dropping.
Rows remaining: 9,407  (was 9,407)

Long-format table: 37,628 rows × 8 columns
is_chosen distribution:
False    28221
True      9407


In [19]:
# ── Cell 21: Diagnostics ──────────────────────────────────────────────────

n_rows_before = len(filt)
n_rows_clean  = len(filt_clean)
n_rows_long   = len(long_df)

print('=== Row counts ===')
print(f'  Original annotator rows:              {n_rows_before:>7,}')
print(f'  After dropping all-4-same:            {n_rows_clean:>7,}  (dropped {n_rows_before - n_rows_clean:,})')
print(f'  Long-format rows (×4 positions):      {n_rows_long:>7,}')
print(f'\n=== Unique values ===')
print(f'  Unique prompts:                       {long_df[COL_PROMPT].nunique():>7,}')
print(f'  Unique annotators:                    {long_df[COL_ANNOTATOR].nunique():>7,}')
print(f'  Unique response texts (approx):       {long_df["response_text"].nunique():>7,}')
print(f'\n=== Chosen-response position balance ===')
print(long_df[long_df['is_chosen']][COL_PREFERRED].value_counts().sort_index().to_string())

=== Row counts ===
  Original annotator rows:                9,407
  After dropping all-4-same:              9,407  (dropped 0)
  Long-format rows (×4 positions):       37,628

=== Unique values ===
  Unique prompts:                         1,008
  Unique annotators:                        598
  Unique response texts (approx):        11,183

=== Chosen-response position balance ===
response_a    3010
response_b    1458
response_c    1562
response_d    3377


In [20]:
# ── Cell 22: Final long-format table ──────────────────────────────────────
#
# This is the table we carry forward. No deduplication.
# One row per annotator × response position.
# Columns needed for feature scoring: first_turn_prompt, response_text.
# Columns kept for validation only:   conversation_id, annotator_id,
#                                     first_turn_preferred_response,
#                                     first_turn_feedback, is_chosen.

resp_table = long_df.rename(columns={COL_PROMPT: 'prompt_text'}).copy()

print(f'resp_table: {len(resp_table):,} rows × {resp_table.shape[1]} columns')
print(f'Columns: {resp_table.columns.tolist()}')
print()
resp_table[['prompt_text', 'response_text', 'response_position', 'is_chosen']].head(8)

resp_table: 37,628 rows × 8 columns
Columns: ['prompt_text', 'conversation_id', 'annotator_id', 'first_turn_preferred_response', 'first_turn_feedback', 'response_text', 'response_position', 'is_chosen']



,prompt_text,response_text,response_position,is_chosen
0,Can you give me some tips for choosing the perfect haircut for my hair type?,"The key to choosing the perfect haircut for your hair type is understanding its unique characteristics. For curly hair, look for cuts that enhance natural texture without too much layering, which can lead to frizz. Straight hair can pull off blunt cuts and sharp lines, adding a sleek, polished l...",response_a,True
1,What are the best local festivals to attend in the Northeast region?,"The Northeast region is home to a diverse array of local festivals that celebrate music, food, and culture. One of the most popular festivals is the Lowell Folk Festival in Massachusetts, which showcases traditional music and dance from around the world. Another notable festival is the Yonkers R...",response_a,True
2,"Write a meaningful welcome message to new colleagues, introducing myself and offering to help them settle into their new roles at our company, specifically highlighting the benefits of our mentorship program and casual team outings.","It's fantastic to have you on board. I'm excited to introduce myself and offer any assistance you might need as you navigate your new role. Our company prides itself on its supportive environment, and a key part of that is our mentorship program, which pairs you with experienced colleagues who c...",response_a,False
3,"I'm a project manager for a construction company and I need help drafting a project plan for a new commercial development, including milestones and deadlines.","The project plan for the new commercial development should prioritize sustainability and environmental responsibility. This includes incorporating green building materials, optimizing energy efficiency, and implementing a waste management system. Milestones should be set for the completion of th...",response_a,False
4,Can you recommend a good desk for a small home office?,"Consider a compact, wall-mounted desk to save floor space and create a sense of openness in your small home office. Look for one with built-in cable management and a sturdy shelf for your computer monitor or laptop. This design will help keep your workspace organized and clutter-free, allowing y...",response_a,True
5,"I need help writing a letter to a client apologizing for a mistake in their order, including a solution and a gesture of goodwill.","A sincere apology is essential in maintaining a positive relationship with the client. The letter should clearly state the mistake, take responsibility for the error, and provide a concrete solution to rectify the situation. Offering a gesture of goodwill, such as a discount on the next order or...",response_a,False
6,"I'm a coffee lover, can you recommend some popular cafes in Pondicherry?","Pondicherry is a haven for coffee lovers, with its rich French colonial history and vibrant cultural scene. Some popular cafes include Cafe Xtasi, which serves a range of specialty coffees and delicious pastries, and Le Cafe, a charming spot with a relaxed atmosphere and stunning views of the Ba...",response_a,False
7,What are the key events of the American Civil War?,"The American Civil War was a pivotal moment in the nation's history, marked by the secession of 11 Southern states and the formation of the Confederate States of America. Key events include the Battle of Fort Sumter, the first battle of the war, and the issuance of the Emancipation Proclamation ...",response_a,False


In [21]:
# ── Cell 23: Scoring rubric — 0 / 1 / 2 ordinal scale ─────────────────────
#
# Scale rationale:
#   0/1/2 is the smallest ordinal scale that can capture:
#     0 = clearly absent / poor
#     1 = partially present / mixed
#     2 = clearly present / good
#   A binary 0/1 loses the "partial" middle that scorers naturally reach for.
#   A 0-4 or 0-10 scale adds precision that is not recoverable from free-text
#   explanations and creates inter-rater disagreement without adding signal.
#   A 0/1/2 scale also makes the feature-score template easy to fill by hand.
#
# Scoring rule: read the prompt, then read THIS response only.
# Do not look at the other three responses or at the preferred label.

rubric = [
    {
        'feature': 'instruction_adherence',
        'definition': 'The response addresses the specific task or question posed in the prompt without ignoring subparts or substituting a different question.',
        'score_0': 'Does not address the main task, or answers a substantially different question.',
        'score_1': 'Addresses the main task but misses or misreads a stated constraint or sub-part.',
        'score_2': 'Directly and fully addresses every part of the prompt as stated.',
        'confusable_with': 'completeness — adherence is about being on-task at all; completeness is about depth of coverage within the task.',
    },
    {
        'feature': 'specificity',
        'definition': 'The response provides concrete details, numbers, named entities, or step-by-step instructions rather than generic statements.',
        'score_0': 'Entirely generic; no concrete detail, numbers, names, or steps anywhere in the response.',
        'score_1': 'Mix of concrete and vague; some useful specifics alongside significant vague claims.',
        'score_2': 'Consistently concrete throughout; uses numbers, proper names, step-by-step instructions, or worked examples.',
        'confusable_with': 'completeness — a response can be long and complete yet still vague; specificity is about concreteness, not length.',
    },
    {
        'feature': 'completeness',
        'definition': 'The response covers all logically necessary parts of the answer given the prompt, leaving no important sub-question unaddressed.',
        'score_0': 'Missing a logically necessary component; a follow-up question would be needed to get a usable answer.',
        'score_1': 'Covers the main point but leaves one secondary aspect underdeveloped or entirely absent.',
        'score_2': 'Covers every facet a reasonable reader would expect; no gaps.',
        'confusable_with': 'instruction_adherence — completeness is about coverage depth; adherence is about being on-target at all.',
    },
    {
        'feature': 'clarity',
        'definition': 'The response is written in language that is easy to parse: vocabulary appropriate to the audience, short sentences, no ambiguous pronoun references.',
        'score_0': 'Hard to parse due to dense jargon, long tangled sentences, or ambiguous references.',
        'score_1': 'Mostly readable but has one or more passages that require re-reading or interpretation.',
        'score_2': 'Easy to read from start to finish; vocabulary matches the apparent audience.',
        'confusable_with': 'coherence — clarity is sentence-level readability; coherence is paragraph/discourse-level logical flow.',
    },
    {
        'feature': 'coherence',
        'definition': 'The response has logical internal structure: ideas are ordered sensibly, transitions are present, no contradictions appear within the text.',
        'score_0': 'Ideas jump around, contradict each other, or repeat without adding new content.',
        'score_1': 'Logical overall but one transition or ordering is awkward or misplaced.',
        'score_2': 'Well-organized; ideas build on each other in natural reading order.',
        'confusable_with': 'clarity — clarity is about word/sentence legibility; coherence is about whether the logical structure of ideas makes sense.',
    },
    {
        'feature': 'conciseness',
        'definition': 'The response delivers needed information without padding, repetition, or off-topic digressions.',
        'score_0': 'Contains substantial padding, repetition, or content that does not serve the prompt.',
        'score_1': 'Mostly focused but includes one or two redundant passages or filler sentences.',
        'score_2': 'Every sentence adds information; no padding.',
        'confusable_with': 'completeness — a concise response can still be complete; the tension only arises when cutting content creates gaps.',
    },
    {
        'feature': 'factual_plausibility',
        'definition': 'The factual claims in the response are consistent with broadly known facts and do not contain obvious errors.',
        'score_0': 'Contains an obvious factual error, impossible claim, or internal factual contradiction.',
        'score_1': 'No clear errors but includes unverified, borderline, or hedged claims that may be wrong.',
        'score_2': 'All stated facts are plausible and consistent with broadly known knowledge.',
        'confusable_with': 'instruction_adherence — a response can be on-topic but factually wrong; these are orthogonal dimensions.',
    },
    {
        'feature': 'tone_fit',
        'definition': 'The formality, warmth, and register of the response match the implied context of the prompt.',
        'score_0': 'Tone clearly mismatches the prompt (e.g., flippant for a serious topic; clinical for an emotional one).',
        'score_1': 'Acceptable tone but slightly miscalibrated — a bit too formal, too casual, or too detached.',
        'score_2': 'Tone naturally matches the register implied by the prompt.',
        'confusable_with': 'safety — tone_fit is about register match; safety is about harm avoidance. A response can be appropriately toned but still unsafe.',
    },
    {
        'feature': 'actionability',
        'definition': 'The response gives the user something concrete they can do next: a specific recommendation, step, command, or decision criterion.',
        'score_0': 'Presents only background or tradeoffs with no concrete next step or recommendation.',
        'score_1': 'Offers some direction but stops short of a specific, concrete recommendation or step.',
        'score_2': 'Gives the user something specific and concrete to do immediately.',
        'confusable_with': 'specificity — actionability requires specificity, but a response can be specific about background facts without being actionable.',
    },
    {
        'feature': 'use_of_examples',
        'definition': 'The response illustrates its claims with at least one concrete example, analogy, or worked demonstration.',
        'score_0': 'No examples, analogies, or worked demonstrations anywhere in the response.',
        'score_1': 'One example is present but it is superficial or only loosely related to the main claim.',
        'score_2': 'One or more directly relevant, well-chosen examples that clearly illustrate the main point.',
        'confusable_with': 'specificity — examples are one way to achieve specificity, but specificity can also come from numbers or names with no example.',
    },
    {
        'feature': 'safety',
        'definition': 'The response avoids harmful, offensive, or inappropriate content.',
        'score_0': 'Contains harmful instructions, offensive language, or clearly inappropriate content.',
        'score_1': 'No harmful content but missing an appropriate disclaimer where one is clearly warranted.',
        'score_2': 'Safe; no harmful content; disclaimers are present and proportionate where needed.',
        'confusable_with': 'tone_fit — safety is about harm avoidance; tone_fit is about register match. A safe response can still have the wrong tone.',
    },
]

rubric_df = pd.DataFrame(rubric)
print(f'Rubric: {len(rubric_df)} features on 0/1/2 scale.\n')
for r in rubric:
    print(f"{'─'*70}")
    print(f"  {r['feature'].upper()}")
    print(f"  Definition: {r['definition']}")
    print(f"  0 → {r['score_0']}")
    print(f"  1 → {r['score_1']}")
    print(f"  2 → {r['score_2']}")
    print(f"  Confusable with: {r['confusable_with']}")
    print()

Rubric: 11 features on 0/1/2 scale.

──────────────────────────────────────────────────────────────────────
  INSTRUCTION_ADHERENCE
  Definition: The response addresses the specific task or question posed in the prompt without ignoring subparts or substituting a different question.
  0 → Does not address the main task, or answers a substantially different question.
  1 → Addresses the main task but misses or misreads a stated constraint or sub-part.
  2 → Directly and fully addresses every part of the prompt as stated.
  Confusable with: completeness — adherence is about being on-task at all; completeness is about depth of coverage within the task.

──────────────────────────────────────────────────────────────────────
  SPECIFICITY
  Definition: The response provides concrete details, numbers, named entities, or step-by-step instructions rather than generic statements.
  0 → Entirely generic; no concrete detail, numbers, names, or steps anywhere in the response.
  1 → Mix of concrete 

In [22]:
# ── Cell 24: JSON schema for scoring one response ─────────────────────────
#
# This schema describes the structure of a single scored record.
# It can be used to validate output from a human scorer or a model scorer
# before merging into the feature-score table.

import json

score_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "ResponseScoreRecord",
    "description": "Scores for one response on all 11 features. Score each feature independently using only the prompt and this response — do NOT use other responses or the preferred label.",
    "type": "object",
    "required": ["response_id", "prompt_id", "scorer", "scores"],
    "properties": {
        "response_id": {
            "type": "string",
            "description": "SHA-256 hex ID of (prompt_text || response_text) — matches resp_table.response_id"
        },
        "prompt_id": {
            "type": "string",
            "description": "SHA-256 hex ID of prompt_text — matches resp_table.prompt_id"
        },
        "scorer": {
            "type": "string",
            "description": "Identifier of the scorer: a human annotator ID, model name, or 'auto'"
        },
        "scored_at": {
            "type": "string",
            "format": "date-time",
            "description": "ISO-8601 timestamp of when scoring was completed"
        },
        "scores": {
            "type": "object",
            "required": FEATURES,
            "additionalProperties": False,
            "properties": {feat: {
                "type": "integer",
                "enum": [0, 1, 2],
                "description": rubric_df.loc[rubric_df['feature'] == feat, 'definition'].iloc[0]
            } for feat in FEATURES},
        },
        "notes": {
            "type": "string",
            "description": "Optional free-text annotation from the scorer"
        }
    },
    "additionalProperties": False,
}

schema_str = json.dumps(score_schema, indent=2)
print(schema_str)

{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "ResponseScoreRecord",
  "description": "Scores for one response on all 11 features. Score each feature independently using only the prompt and this response \u2014 do NOT use other responses or the preferred label.",
  "type": "object",
  "required": [
    "response_id",
    "prompt_id",
    "scorer",
    "scores"
  ],
  "properties": {
    "response_id": {
      "type": "string",
      "description": "SHA-256 hex ID of (prompt_text || response_text) \u2014 matches resp_table.response_id"
    },
    "prompt_id": {
      "type": "string",
      "description": "SHA-256 hex ID of prompt_text \u2014 matches resp_table.prompt_id"
    },
    "scorer": {
      "type": "string",
      "description": "Identifier of the scorer: a human annotator ID, model name, or 'auto'"
    },
    "scored_at": {
      "type": "string",
      "format": "date-time",
      "description": "ISO-8601 timestamp of when scoring was completed"
    },

In [24]:
# ── Cell 25: Feature-score template dataframe ─────────────────────────────
#
# One row per annotator × response position (same index as resp_table).
# conversation_id + response_position uniquely identify each row.
# Feature columns are initialized to pd.NA — not yet scored.

score_template = resp_table[[
    COL_CONV_ID, COL_ANNOTATOR, 'response_position', 'is_chosen',
]].copy().rename(columns={
    COL_CONV_ID:   'conversation_id',
    COL_ANNOTATOR: 'annotator_id',
})

# Short previews so scorers can see what they're rating
score_template['prompt_preview']   = resp_table['prompt_text'].str[:100]
score_template['response_preview'] = resp_table['response_text'].str[:100]

# 11 feature columns — nullable Int8, NA until scored
for feat in FEATURES:
    score_template[feat] = pd.array([pd.NA] * len(score_template), dtype='Int8')

score_template['scorer'] = ''
score_template['notes']  = ''

print(f'Score template: {len(score_template):,} rows × {score_template.shape[1]} columns')
print(f'Columns: {score_template.columns.tolist()}')
print(f'\nNA count per feature (should all be {len(score_template):,}):')
print(score_template[FEATURES].isna().sum().to_string())
score_template.head(4)

Score template: 37,628 rows × 19 columns
Columns: ['conversation_id', 'annotator_id', 'response_position', 'is_chosen', 'prompt_preview', 'response_preview', 'instruction_adherence', 'specificity', 'completeness', 'clarity', 'coherence', 'conciseness', 'factual_plausibility', 'tone_fit', 'actionability', 'use_of_examples', 'safety', 'scorer', 'notes']

NA count per feature (should all be 37,628):
instruction_adherence    37628
specificity              37628
completeness             37628
clarity                  37628
coherence                37628
conciseness              37628
factual_plausibility     37628
tone_fit                 37628
actionability            37628
use_of_examples          37628
safety                   37628


,conversation_id,annotator_id,response_position,is_chosen,prompt_preview,response_preview,instruction_adherence,specificity,completeness,clarity,coherence,conciseness,factual_plausibility,tone_fit,actionability,use_of_examples,safety,scorer,notes
0,702992025732060,61575131153481,response_a,True,Can you give me some tips for choosing the perfect haircut for my hair type?,The key to choosing the perfect haircut for your hair type is understanding its unique characteristi,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
1,695771192796939,61574767872008,response_a,True,What are the best local festivals to attend in the Northeast region?,"The Northeast region is home to a diverse array of local festivals that celebrate music, food, and c",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
2,654427437500427,61575257361849,response_a,False,"Write a meaningful welcome message to new colleagues, introducing myself and offering to help them s",It's fantastic to have you on board. I'm excited to introduce myself and offer any assistance you mi,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
3,1041030501272902,61574928906639,response_a,False,I'm a project manager for a construction company and I need help drafting a project plan for a new c,The project plan for the new commercial development should prioritize sustainability and environment,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,


In [25]:
# ── Cell 26: Save artifacts ────────────────────────────────────────────────

OUT_DIR = 'empirics_communityalignment'
os.makedirs(OUT_DIR, exist_ok=True)

def save(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_parquet(path, index=False)
    print(f'  {name:45s}  {len(df):>7,} rows  ({os.path.getsize(path)/1e3:.0f} KB)')

print('Saving parquet files:')
save(resp_table, 'responses.parquet')

rubric_csv  = os.path.join(OUT_DIR, 'scoring_rubric.csv')
rubric_json = os.path.join(OUT_DIR, 'scoring_rubric.json')
schema_path = os.path.join(OUT_DIR, 'score_record_schema.json')
rubric_df.to_csv(rubric_csv, index=False)
with open(rubric_json, 'w') as f:  json.dump(rubric, f, indent=2)
with open(schema_path, 'w') as f:  json.dump(score_schema, f, indent=2)
print(f'\n  scoring_rubric.csv / .json')
print(f'  score_record_schema.json')

template_csv     = os.path.join(OUT_DIR, 'feature_score_template_blank.csv')
template_parquet = os.path.join(OUT_DIR, 'feature_score_template_blank.parquet')
score_template.to_csv(template_csv, index=False, na_rep='')
score_template.to_parquet(template_parquet, index=False)
print(f'  feature_score_template_blank.csv / .parquet')

print(f'\nDone. responses.parquet has {len(resp_table):,} rows ready for scoring.')

Saving parquet files:
  responses.parquet                               37,628 rows  (20876 KB)

  scoring_rubric.csv / .json
  score_record_schema.json
  feature_score_template_blank.csv / .parquet

Done. responses.parquet has 37,628 rows ready for scoring.


---
## Part 4 — Model Scorer

Use the Anthropic **Batches API** to fill in the 11-feature score template for all 37,628 response rows.

**Design choices:**
- **Batches API** (`client.messages.batches`): async processing, 50% cost discount vs. synchronous calls
- **Prompt caching** on the system block: the ~1,500-token rubric is cached after the first request, cutting per-request input cost to ~0.1×
- **Structured output** (`output_config.format`): guarantees a valid `{feature: 0|1|2}` JSON object, no post-hoc parsing failures
- **Checkpoint file**: saves the batch ID to disk so a kernel restart cannot submit a duplicate batch

**Scoring rule**: only `prompt_text` and `response_text` are passed to the model. `is_chosen`, `first_turn_feedback`, and `response_position` are audit-only and are never sent.

In [34]:
# ── Cell 27: Anthropic SDK — imports and client setup ─────────────────────
#
# Cost note (Batches API = 50% discount off list price):
#   claude-opus-4-7  ~$2.50/M cached input + $12.50/M output
#   37k rows × ~600 input tokens ≈ 22M tokens ≈ $55 input (mostly cached)
#   To reduce cost, change MODEL to 'claude-haiku-4-5' (~$0.50/M input).

import anthropic
import json, os, time, datetime
from pathlib import Path

# Load .env from the project root (no extra packages needed)
_env_path = Path('.env')
if _env_path.exists():
    for _line in _env_path.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith('#') and '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k.strip(), _v.strip())

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

MODEL         = 'claude-opus-4-7'
BATCH_ID_FILE = 'empirics_communityalignment/scorer_batch_id.txt'
SCORED_PATH   = 'empirics_communityalignment/feature_scores_model.parquet'

print(f'anthropic SDK: {anthropic.__version__}')
print(f'Model:         {MODEL}')
print(f'Rows to score: {len(resp_table):,}')
print(f'API key loaded: {"yes" if os.environ.get("ANTHROPIC_API_KEY") else "NO — check .env"}')


anthropic SDK: 0.96.0
Model:         claude-opus-4-7
Rows to score: 37,628
API key loaded: yes


In [35]:
# ── Cell 28: Build scoring system prompt with prompt caching ──────────────
#
# The rubric is identical for all 37k requests — ideal for prompt caching.
# cache_control: {"type": "ephemeral"} on the system block tells Anthropic
# to cache the processed KV state after the first request in the batch.
# Subsequent requests pay ~0.1× the normal input cost for the cached tokens.
# Minimum cacheable prefix: 1024 tokens. This rubric is well above that.

def build_rubric_text(rubric_list):
    lines = [
        "You are a response quality scorer. Score the RESPONSE on each of the "
        "11 features below using a 0/1/2 ordinal scale based solely on the "
        "PROMPT and RESPONSE provided. Do not use information about other "
        "responses or the annotator's preferred choice.",
        "",
        "SCALE:  0 = clearly absent / poor   |   1 = partially present / mixed   |   2 = clearly present / good",
        "",
        "FEATURES AND CRITERIA:",
    ]
    for r in rubric_list:
        lines.append(f"\n## {r['feature'].upper()}")
        lines.append(f"Definition: {r['definition']}")
        lines.append(f"  Score 0: {r['score_0']}")
        lines.append(f"  Score 1: {r['score_1']}")
        lines.append(f"  Score 2: {r['score_2']}")
        lines.append(f"  Confusable with: {r['confusable_with']}")
    lines += [
        "",
        "OUTPUT: Return ONLY a JSON object with exactly these 11 integer keys (values in {0, 1, 2}):",
        '{"instruction_adherence": ?, "specificity": ?, "completeness": ?, "clarity": ?, '
        '"coherence": ?, "conciseness": ?, "factual_plausibility": ?, "tone_fit": ?, '
        '"actionability": ?, "use_of_examples": ?, "safety": ?}',
        "No prose, no explanation.",
    ]
    return "\n".join(lines)

SYSTEM_TEXT = build_rubric_text(rubric)

# system prompt as a cacheable content block
SYSTEM_BLOCK = {
    "type": "text",
    "text": SYSTEM_TEXT,
    "cache_control": {"type": "ephemeral"},
}

n_chars      = len(SYSTEM_TEXT)
n_tokens_est = n_chars // 4
print(f'System prompt: {n_chars:,} chars  (~{n_tokens_est:,} tokens est.)')
above = n_tokens_est >= 1024
print(f'Prompt caching: {"active — above 1024-token threshold" if above else "WARNING — may be below 1024-token threshold"}')
print(f'\n--- System prompt preview (first 500 chars) ---')
print(SYSTEM_TEXT[:500])

System prompt: 6,866 chars  (~1,716 tokens est.)
Prompt caching: active — above 1024-token threshold

--- System prompt preview (first 500 chars) ---
You are a response quality scorer. Score the RESPONSE on each of the 11 features below using a 0/1/2 ordinal scale based solely on the PROMPT and RESPONSE provided. Do not use information about other responses or the annotator's preferred choice.

SCALE:  0 = clearly absent / poor   |   1 = partially present / mixed   |   2 = clearly present / good

FEATURES AND CRITERIA:

## INSTRUCTION_ADHERENCE
Definition: The response addresses the specific task or question posed in the prompt without ignori


In [36]:
# ── Cell 29: Build batch request list ─────────────────────────────────────
#
# One request per row in resp_table (37,628 rows).
# custom_id = "{conversation_id}_{response_position}" — unique per row,
# reversible back to the resp_table index without a separate lookup.
#
# Scoring rule: only prompt_text and response_text are sent to the model.
# is_chosen, feedback, and response_position are NOT passed.

SCORES_SCHEMA = {
    "type": "object",
    "properties": {feat: {"type": "integer", "enum": [0, 1, 2]} for feat in FEATURES},
    "required": FEATURES,
    "additionalProperties": False,
}

def build_user_message(prompt_text, response_text):
    return (
        f"PROMPT:\n{prompt_text}\n\n"
        f"RESPONSE TO SCORE:\n{response_text}"
    )

requests = []
for _, row in resp_table.iterrows():
    custom_id = f"{int(row['conversation_id'])}_{row['response_position']}"
    requests.append({
        "custom_id": custom_id,
        "params": {
            "model":      MODEL,
            "max_tokens": 256,
            "system":     [SYSTEM_BLOCK],
            "messages":   [{"role": "user", "content": build_user_message(
                                row['prompt_text'], row['response_text'])}],
            "output_config": {
                "format": {
                    "type":   "json_schema",
                    "schema": SCORES_SCHEMA,
                }
            },
        },
    })

print(f'Built {len(requests):,} batch requests.')
s = requests[0]
print(f'\nSample request:')
print(f'  custom_id:   {s["custom_id"]}')
print(f'  model:       {s["params"]["model"]}')
print(f'  max_tokens:  {s["params"]["max_tokens"]}')
print(f'  user message (first 250 chars):')
print(f'    {s["params"]["messages"][0]["content"][:250]}')

Built 37,628 batch requests.

Sample request:
  custom_id:   702992025732060_response_a
  model:       claude-opus-4-7
  max_tokens:  256
  user message (first 250 chars):
    PROMPT:
Can you give me some tips for choosing the perfect haircut for my hair type?

RESPONSE TO SCORE:
The key to choosing the perfect haircut for your hair type is understanding its unique characteristics. For curly hair, look for cuts that enhanc


In [38]:
# ── Cell 30: Submit batches or resume from checkpoint ─────────────────────
#
# Set N_SAMPLE to an integer to run a sanity-check subset, or None for all.

N_SAMPLE      = 2_000   # ← change to None for the full 37k run
CHUNK_SIZE    = 5_000
BATCH_ID_PATH = Path(BATCH_ID_FILE)

requests_to_send = requests[:N_SAMPLE] if N_SAMPLE else requests
print(f'Requests to send: {len(requests_to_send):,}  (of {len(requests):,} total)')

if BATCH_ID_PATH.exists():
    batch_ids = json.loads(BATCH_ID_PATH.read_text())
    print(f'Resumed {len(batch_ids)} batch(es) from checkpoint.')
else:
    chunks = [requests_to_send[i:i+CHUNK_SIZE] for i in range(0, len(requests_to_send), CHUNK_SIZE)]
    print(f'Splitting into {len(chunks)} batch(es) of ≤{CHUNK_SIZE:,}...')
    batch_ids = []
    for k, chunk in enumerate(chunks):
        b = client.messages.batches.create(requests=chunk)
        batch_ids.append(b.id)
        print(f'  Batch {k+1}/{len(chunks)}: {b.id}  ({len(chunk):,} requests)')
    BATCH_ID_PATH.parent.mkdir(parents=True, exist_ok=True)
    BATCH_ID_PATH.write_text(json.dumps(batch_ids))
    print(f'\nAll IDs saved → {BATCH_ID_FILE}')

print('\nCurrent status:')
for bid in batch_ids:
    b = client.messages.batches.retrieve(bid)
    print(f'  {bid}  {b.processing_status}  {b.request_counts}')


Requests to send: 2,000  (of 37,628 total)
Splitting into 1 batch(es) of ≤5,000...
  Batch 1/1: msgbatch_01W7Jiov8wjZCFog5mWLGoen  (2,000 requests)

All IDs saved → empirics_communityalignment/scorer_batch_id.txt

Current status:
  msgbatch_01W7Jiov8wjZCFog5mWLGoen  in_progress  MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2000, succeeded=0)


In [39]:
# ── Cell 31: Poll until all batches end ───────────────────────────────────

POLL_INTERVAL_S = 60

print(f'Polling {len(batch_ids)} batch(es) every {POLL_INTERVAL_S}s...')
while True:
    statuses = []
    for bid in batch_ids:
        b = client.messages.batches.retrieve(bid)
        statuses.append(b.processing_status)
    ts    = datetime.datetime.now().strftime('%H:%M:%S')
    ended = sum(s == 'ended' for s in statuses)
    print(f'  [{ts}]  {ended}/{len(batch_ids)} ended')
    if all(s == 'ended' for s in statuses):
        break
    time.sleep(POLL_INTERVAL_S)

print('\nAll batches complete.')
for bid in batch_ids:
    b = client.messages.batches.retrieve(bid)
    print(f'  {bid}  succeeded={b.request_counts.succeeded}  errored={b.request_counts.errored}')


Polling 1 batch(es) every 60s...
  [13:55:01]  0/1 ended
  [13:56:01]  0/1 ended
  [13:57:01]  0/1 ended
  [13:58:01]  1/1 ended

All batches complete.
  msgbatch_01W7Jiov8wjZCFog5mWLGoen  succeeded=2000  errored=0


In [40]:
# ── Cell 32: Parse results from all batches into score_template ────────────

print(f'Fetching results from {len(batch_ids)} batch(es)...')

scores_lookup = {}   # custom_id → {feature: int}
error_ids     = []

for bid in batch_ids:
    for result in client.messages.batches.results(bid):
        cid = result.custom_id
        if result.result.type == 'succeeded':
            try:
                text   = result.result.message.content[0].text
                parsed = json.loads(text)
            except Exception:
                parsed = {}
            scores_lookup[cid] = parsed
        else:
            error_ids.append(cid)

print(f'  Parsed succeeded: {len(scores_lookup):,}')
print(f'  Errors:           {len(error_ids):,}')
if error_ids:
    print(f'  Sample error IDs: {error_ids[:5]}')

# ── Fill score_template ───────────────────────────────────────────────────
scored      = score_template.copy()
SCORER_NAME = MODEL

n_filled  = 0
n_missing = 0

for idx, row in resp_table.iterrows():
    cid    = f"{int(row['conversation_id'])}_{row['response_position']}"
    scores = scores_lookup.get(cid)
    if scores is None:
        n_missing += 1
        continue
    for feat in FEATURES:
        val = scores.get(feat)
        if val in (0, 1, 2):
            scored.at[idx, feat] = val
    scored.at[idx, 'scorer'] = SCORER_NAME
    n_filled += 1

print(f'\nRows filled: {n_filled:,}  |  Missing: {n_missing:,}')
print('\nScore distributions per feature:')
print(scored[FEATURES].apply(lambda s: s.value_counts().sort_index()).T.fillna(0).astype(int).to_string())


Fetching results from 1 batch(es)...
  Parsed succeeded: 2,000
  Errors:           0

Rows filled: 2,000  |  Missing: 35,628

Score distributions per feature:
                         0     1     2
instruction_adherence   19   511  1470
specificity            265  1121   614
completeness           240  1476   284
clarity                  0     7  1993
coherence                0    29  1971
conciseness              2   179  1819
factual_plausibility    16   346  1638
tone_fit                 3   252  1745
actionability          275  1314   411
use_of_examples        616   751   633
safety                   0    53  1947


In [41]:
# ── Cell 33: Save scored feature table ────────────────────────────────────

scored.to_parquet(SCORED_PATH, index=False)
print(f'Saved → {SCORED_PATH}')
print(f'  Rows:    {len(scored):,}')
print(f'  Scored:  {scored["scorer"].ne("").sum():,}')
print(f'  Columns: {scored.columns.tolist()}')

# Mean score per feature (scored rows only)
print('\nMean score per feature (scored rows only):')
mask = scored['scorer'].ne('')
print(scored.loc[mask, FEATURES].mean().round(3).to_string())

Saved → empirics_communityalignment/feature_scores_model.parquet
  Rows:    37,628
  Scored:  2,000
  Columns: ['conversation_id', 'annotator_id', 'response_position', 'is_chosen', 'prompt_preview', 'response_preview', 'instruction_adherence', 'specificity', 'completeness', 'clarity', 'coherence', 'conciseness', 'factual_plausibility', 'tone_fit', 'actionability', 'use_of_examples', 'safety', 'scorer', 'notes']

Mean score per feature (scored rows only):
instruction_adherence    1.726
specificity              1.174
completeness             1.022
clarity                  1.996
coherence                1.986
conciseness              1.908
factual_plausibility     1.811
tone_fit                 1.871
actionability            1.068
use_of_examples          1.008
safety                   1.974
